# Modelos Supervisados — Clasificación

Notebook dedicado al entrenamiento y evaluación de modelos de clasificación supervisada.

**Modelos seleccionados:**
- Bagging
- Gradient Boosting
- Random Forest
- AdaBoost
- SVM con Kernel RBF

**Preprocesado:**
1. **Selección de características**: eliminación de variables con correlación absoluta > 0.99 (una por cada par).
2. **Estandarización**: `StandardScaler` por las grandes diferencias de escalas y rangos.
3. **Desbalanceo de clases**: `class_weight='balanced'` en los modelos que lo soporten.

## 1. Configuración e importaciones

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Modelos
from sklearn.ensemble import (
    BaggingClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

SEED = 42
np.random.seed(SEED)

print("Imports OK")

Imports OK


## 2. Carga de datos

In [2]:
train_df = pd.read_csv('data/training.csv', index_col=0)
test_df  = pd.read_csv('data/test.csv', index_col=0)

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")
train_df.head()

Train: (3000, 51)
Test:  (2000, 51)


,bqwyz,glhls,gwrec,bpsmt,csuhz,dsdfq,dhmrb,qimhg,dnywb,bbznq,...,dbcsa,vieby,mpqtr,vxswx,xpwzl,vbsda,fiptu,jahkp,ownqd,class
1,-12.993810,-3.689143,6.000694,17.215317,15.319216,0.003135,17.779791,16.248288,0.253989,-17.427260,...,0.616191,-1.037537,0.640160,-1.586657,1.786693,0.854592,-1.318690,-0.317202,1.387131,1
2,6.246265,1.764485,-3.318136,15.920140,15.471565,0.012288,-8.952036,16.345302,1.098853,-4.312381,...,-1.436535,0.439239,0.298879,-0.401065,1.441416,-0.617473,-0.137313,1.982024,1.249828,0
3,-3.039952,-0.262428,1.168803,-13.059538,-13.989008,0.022966,2.920655,-12.881768,0.795419,3.261357,...,-1.371939,1.531874,-0.156040,-0.795904,0.114599,-0.416141,0.050341,0.200234,1.222475,0
4,-10.644402,-3.330536,5.633811,-11.523825,-11.433825,0.030853,14.704378,-11.115733,1.379458,-16.017284,...,0.060147,0.547056,-1.044690,0.636920,0.376566,-1.013476,-0.710287,1.150738,0.391557,0
5,-4.488924,-1.091610,2.512258,1.073196,2.646280,0.027421,5.430353,2.135076,1.212872,-28.630766,...,-1.371114,2.015222,1.064987,3.702119,1.460052,1.066106,-0.798662,1.337375,1.524883,1


## 3. Selección de características — Eliminación de variables altamente correlacionadas

Del análisis exploratorio se identificaron parejas de variables con |correlación| > 0.99:

| Variable 1 | Variable 2 | Correlación |
|:----------:|:----------:|:-----------:|
| qimhg      | bpsmt      |  0.9995     |
| qimhg      | csuhz      |  0.9994     |
| csuhz      | bpsmt      |  0.9988     |
| dhmrb      | gwrec      |  0.9954     |
| dhmrb      | glhls      | −0.9952     |
| dhmrb      | bqwyz      | −0.9951     |
| gwrec      | glhls      | −0.9905     |
| gwrec      | bqwyz      | −0.9904     |
| glhls      | bqwyz      |  0.9903     |

Se forman dos clusters de variables "gemelas":
- **Cluster 1**: `qimhg`, `bpsmt`, `csuhz` → conservamos **`qimhg`**, eliminamos `bpsmt` y `csuhz`.
- **Cluster 2**: `dhmrb`, `gwrec`, `glhls`, `bqwyz` → conservamos **`dhmrb`**, eliminamos `gwrec`, `glhls` y `bqwyz`.

In [3]:
# Variables a eliminar (una por cada par de gemelas, conservamos una representante)
cols_to_drop = ['bpsmt', 'csuhz', 'gwrec', 'glhls', 'bqwyz']

TARGET = 'class'

# Separar features y target
X_train = train_df.drop(columns=[TARGET] + cols_to_drop)
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET] + cols_to_drop)
y_test = test_df[TARGET]

print(f"Features tras selección: {X_train.shape[1]} (eliminadas {len(cols_to_drop)})")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"\nDistribución de clases (train):")
print(y_train.value_counts().sort_index())

Features tras selección: 45 (eliminadas 5)
X_train: (3000, 45), X_test: (2000, 45)

Distribución de clases (train):
class
0    2060
1     921
2      19
Name: count, dtype: int64


## 4. Definición de Pipelines

Cada pipeline integra:
1. `StandardScaler` — estandarización de las features.
2. El modelo de clasificación con `class_weight='balanced'` cuando está disponible.

> **Nota sobre Bagging y AdaBoost**: estos meta-estimadores no exponen `class_weight` directamente,
> pero podemos pasar un estimador base que sí lo soporte (e.g. `DecisionTreeClassifier(class_weight='balanced')`).

In [4]:
# Estimador base con class_weight para Bagging y AdaBoost
base_tree = DecisionTreeClassifier(class_weight='balanced', random_state=SEED)

pipelines = {
    'Bagging': Pipeline([
        ('scaler', StandardScaler()),
        ('model', BaggingClassifier(
            estimator=base_tree,
            n_estimators=100,
            random_state=SEED
        ))
    ]),

    'GradientBoosting': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingClassifier(
            n_estimators=100,
            random_state=SEED
            # GradientBoosting no soporta class_weight;
            # se compensa con sample_weight en el fit (ver más abajo)
        ))
    ]),

    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(
            n_estimators=100,
            class_weight='balanced',
            random_state=SEED
        ))
    ]),

    'AdaBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('model', AdaBoostClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=1, class_weight='balanced', random_state=SEED
            ),
            n_estimators=100,
            random_state=SEED
        ))
    ]),

    'SVM_RBF': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(
            kernel='rbf',
            class_weight='balanced',
            random_state=SEED
        ))
    ]),
}

print(f"Pipelines definidos: {list(pipelines.keys())}")

Pipelines definidos: ['Bagging', 'GradientBoosting', 'RandomForest', 'AdaBoost', 'SVM_RBF']


### 4.1 Sample weights para GradientBoosting

`GradientBoostingClassifier` no soporta `class_weight`, así que calculamos `sample_weight`
de forma equivalente a `balanced` para pasarlo durante el `fit`.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights_train = compute_sample_weight('balanced', y_train)

print("Sample weights calculados (primeros 10):")
print(sample_weights_train[:10])

## 5. Validación cruzada estratificada (sobre train)

Evaluamos cada pipeline con CV estratificado de 5 folds para comparar rendimiento antes de ajustar sobre test.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results = {}

for name, pipe in pipelines.items():
    print(f"Evaluando {name}...", end=" ")

    if name == 'GradientBoosting':
        # Para GradientBoosting usamos sample_weight a través de fit_params
        scores = cross_val_score(
            pipe, X_train, y_train, cv=cv, scoring='accuracy',
            fit_params={'model__sample_weight': sample_weights_train}
        )
    else:
        scores = cross_val_score(
            pipe, X_train, y_train, cv=cv, scoring='accuracy'
        )

    cv_results[name] = scores
    print(f"Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

print("\n--- Resumen CV ---")
cv_summary = pd.DataFrame({
    'Mean Accuracy': {k: v.mean() for k, v in cv_results.items()},
    'Std': {k: v.std() for k, v in cv_results.items()}
}).sort_values('Mean Accuracy', ascending=False)
display(cv_summary)

## 6. Entrenamiento final y evaluación sobre test

In [ ]:
test_results = {}

for name, pipe in pipelines.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    # Entrenar
    if name == 'GradientBoosting':
        pipe.fit(X_train, y_train, model__sample_weight=sample_weights_train)
    else:
        pipe.fit(X_train, y_train)

    # Predecir
    y_pred = pipe.predict(X_test)

    # Métricas
    acc = accuracy_score(y_test, y_pred)
    test_results[name] = acc
    print(f"\nAccuracy en test: {acc:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

## 7. Matrices de confusión

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, pipe) in enumerate(pipelines.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
    axes[idx].set_title(name, fontsize=13, fontweight='bold')

# Ocultar el subplot sobrante
axes[-1].set_visible(False)

plt.suptitle('Matrices de Confusión — Datos de Test', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Comparativa final de Accuracy

In [ ]:
results_df = pd.DataFrame({
    'Modelo': list(test_results.keys()),
    'Accuracy (Test)': list(test_results.values())
}).sort_values('Accuracy (Test)', ascending=False).reset_index(drop=True)

display(results_df)

# Gráfico de barras
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(results_df)))
bars = ax.barh(results_df['Modelo'], results_df['Accuracy (Test)'], color=colors)
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('Comparativa de Accuracy en Test', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1)

# Anotaciones
for bar, val in zip(bars, results_df['Accuracy (Test)']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

best = results_df.iloc[0]
print(f"\n✅ Mejor modelo: {best['Modelo']} con Accuracy = {best['Accuracy (Test)']:.4f}")